# ✈️ Flight Deal Finder
Bot para encontrar, comparar e (futuramente) monitorar preços de passagens aéreas.

**Versão atual:** V1 — primeira consulta
**Fonte de dados:** Aviasales/Travelpayouts Data API

In [1]:
# 02 — Dependências
!pip install requests pandas -q

In [2]:
# 03 — Credenciais
from google.colab import userdata

API_TOKEN = userdata.get('TRAVELPAYOUTS_TOKEN')

print("Token carregado:", "OK" if API_TOKEN else "FALHOU")
print("Tamanho do token:", len(API_TOKEN) if API_TOKEN else 0)

Token carregado: OK
Tamanho do token: 32


In [3]:
# 04 — Configuração da API
BASE_URL = "https://api.travelpayouts.com/aviasales/v3/prices_for_dates"

HEADERS = {
    "x-access-token": API_TOKEN
}

In [4]:
# 05 — Busca de voos (função reutilizável)
import pandas as pd
import requests
import unicodedata

resposta_companhias = requests.get("https://api.travelpayouts.com/data/en/airlines.json")
MAPA_COMPANHIAS = {c["code"]: c["name"] for c in resposta_companhias.json()}

def nome_companhia(codigo):
    return MAPA_COMPANHIAS.get(codigo, codigo)

resposta_cidades = requests.get("https://api.travelpayouts.com/data/pt/cities.json")
MAPA_CIDADES = {}
for c in resposta_cidades.json():
    if c.get("name"):
        chave = unicodedata.normalize('NFKD', c["name"].strip().lower())
        chave = ''.join(ch for ch in chave if not unicodedata.combining(ch))
        MAPA_CIDADES[chave] = c["code"]

def resolver_local(texto):
    """Aceita nome de cidade ('São Paulo') ou código IATA ('GRU'/'SAO') e devolve o código."""
    texto = texto.strip()
    if len(texto) == 3 and texto.isalpha():
        return texto.upper()
    chave = unicodedata.normalize('NFKD', texto.lower())
    chave = ''.join(ch for ch in chave if not unicodedata.combining(ch))
    return MAPA_CIDADES.get(chave)

def formatar_duracao(minutos):
    horas = minutos // 60
    mins = minutos % 60
    return f"{horas}h{mins:02d}"

def buscar_voos(origin, destination, departure_at, currency="brl", market="br", sorting="price", limit=30):
    params = {
        "origin": origin,
        "destination": destination,
        "departure_at": departure_at,
        "currency": currency,
        "market": market,
        "sorting": sorting,
        "limit": limit,
    }

    response = requests.get(BASE_URL, headers=HEADERS, params=params)
    response.raise_for_status()
    resultado = response.json()

    voos_tratados = []
    for voo in resultado.get("data", []):
        voos_tratados.append({
            "companhia": nome_companhia(voo["airline"]),
            "origem": voo["origin_airport"],
            "destino": voo["destination_airport"],
            "partida": voo["departure_at"],
            "preco": voo["price"],
            "moeda": resultado["currency"],
            "escalas": voo["transfers"],
            "duracao_min": voo["duration"],
            "duracao": formatar_duracao(voo["duration"]),
            "link": f"https://www.aviasales.com{voo['link']}",
        })

    df = pd.DataFrame(voos_tratados)
    if not df.empty:
        df["partida"] = pd.to_datetime(df["partida"])
        df = df.sort_values("preco").reset_index(drop=True)

    return df
def formatar_preco(valor, moeda="brl"):
    texto = f"{valor:,.2f}"
    texto = texto.replace(",", "X").replace(".", ",").replace("X", ".")
    if moeda.lower() == "brl":
        return f"R$ {texto}"
    return f"{moeda.upper()} {texto}"

In [5]:
# 07
def buscar_datas_flexiveis(origin, destination, data_inicio, data_fim, currency="brl", market="br"):
    data_inicio = pd.to_datetime(data_inicio).date()
    data_fim = pd.to_datetime(data_fim).date()

    meses = pd.period_range(data_inicio, data_fim, freq="M")

    dfs = []
    for mes in meses:
        departure_at = mes.strftime("%Y-%m")
        df_mes = buscar_voos(origin, destination, departure_at, currency=currency, market=market)
        if not df_mes.empty:
            dfs.append(df_mes)

    if not dfs:
        return pd.DataFrame()

    df_completo = pd.concat(dfs, ignore_index=True)
    mask = (df_completo["partida"].dt.date >= data_inicio) & (df_completo["partida"].dt.date <= data_fim)
    return df_completo[mask].sort_values("preco").reset_index(drop=True)

In [6]:
buscar_datas_flexiveis("GRU", "MXP", "2026-09-15", "2026-09-30")

,companhia,origem,destino,partida,preco,moeda,escalas,duracao_min,duracao,link
0,TAP Air Portugal,GRU,MXP,2026-09-24 15:30:00-03:00,4120,brl,1,870,14h30,https://www.aviasales.com/search/GRU2409MXP1?t...
1,TAP Air Portugal,GRU,MXP,2026-09-25 00:45:00-03:00,4120,brl,1,1755,29h15,https://www.aviasales.com/search/GRU2509MXP1?t...
2,TAP Air Portugal,GRU,MXP,2026-09-26 20:45:00-03:00,4120,brl,1,1385,23h05,https://www.aviasales.com/search/GRU2609MXP1?t...
3,TAP Air Portugal,GRU,MXP,2026-09-27 20:45:00-03:00,4120,brl,1,880,14h40,https://www.aviasales.com/search/GRU2709MXP1?t...


In [7]:
#08
def buscar_aeroportos_alternativos(origin, destinos, data_inicio, data_fim, currency="usd", market="br"):
    """
    destinos: lista de códigos IATA alternativos (ex: ["MXP", "LIN", "BGY"])
    Busca em cada aeroporto de destino e devolve tudo junto, ordenado por preço.
    """
    dfs = []
    for destino in destinos:
        df_destino = buscar_datas_flexiveis(origin, destino, data_inicio, data_fim, currency=currency, market=market)
        if not df_destino.empty:
            dfs.append(df_destino)

    if not dfs:
        return pd.DataFrame()

    df_completo = pd.concat(dfs, ignore_index=True)
    return df_completo.sort_values("preco").reset_index(drop=True)

In [8]:
aeroportos_milao = ["MXP", "LIN", "BGY"]
buscar_aeroportos_alternativos("GRU", aeroportos_milao, "2026-10-15", "2026-10-30")

,companhia,origem,destino,partida,preco,moeda,escalas,duracao_min,duracao,link
0,Air China,GRU,MXP,2026-10-26 08:30:00-03:00,366,usd,1,1170,19h30,https://www.aviasales.com/search/GRU2610MXP1?t...
1,TAP Air Portugal,GRU,MXP,2026-10-15 15:30:00-03:00,374,usd,1,1195,19h55,https://www.aviasales.com/search/GRU1510MXP1?t...
2,TAP Air Portugal,GRU,MXP,2026-10-30 01:30:00-03:00,375,usd,1,1105,18h25,https://www.aviasales.com/search/GRU3010MXP1?t...
3,TAP Air Portugal,GRU,MXP,2026-10-16 00:45:00-03:00,376,usd,1,1140,19h00,https://www.aviasales.com/search/GRU1610MXP1?t...
4,Air China,GRU,BGY,2026-10-26 08:30:00-03:00,386,usd,1,1390,23h10,https://www.aviasales.com/search/GRU2610BGY1?t...
5,Air China,GRU,LIN,2026-10-26 08:30:00-03:00,413,usd,3,2005,33h25,https://www.aviasales.com/search/GRU2610LIN1?t...
6,ITA Airways,GRU,LIN,2026-10-21 14:15:00-03:00,421,usd,1,835,13h55,https://www.aviasales.com/search/GRU2110LIN1?t...
7,ITA Airways,GRU,LIN,2026-10-20 18:55:00-03:00,421,usd,1,1035,17h15,https://www.aviasales.com/search/GRU2010LIN1?t...
8,ITA Airways,GRU,LIN,2026-10-22 14:15:00-03:00,422,usd,1,895,14h55,https://www.aviasales.com/search/GRU2210LIN1?t...
9,LATAM Airlines Group,GRU,BGY,2026-10-17 23:50:00-03:00,423,usd,1,1680,28h00,https://www.aviasales.com/search/GRU1710BGY1?t...


In [9]:
# 09
PESOS_PADRAO = {
    "preco": 0.50,
    "duracao": 0.25,
    "escalas": 0.15,
    "conveniencia": 0.10,
}

In [10]:
# 10
def normalizar_invertido(serie):
    """Quanto menor o valor original, maior a nota (0 a 1)."""
    minimo, maximo = serie.min(), serie.max()
    if maximo == minimo:
        return pd.Series(1.0, index=serie.index)
    return 1 - (serie - minimo) / (maximo - minimo)

def calcular_conveniencia(hora):
    """Nota de 0 a 1: voos entre 8h e 20h são os mais convenientes."""
    if 8 <= hora <= 20:
        return 1.0
    dist_8 = min(abs(hora - 8), 24 - abs(hora - 8))
    dist_20 = min(abs(hora - 20), 24 - abs(hora - 20))
    distancia = min(dist_8, dist_20)
    return max(0.0, 1 - distancia / 12)

def calcular_ranking(df, pesos=None):
    if pesos is None:
        pesos = PESOS_PADRAO

    df = df.copy()
    df["preco_score"] = normalizar_invertido(df["preco"])
    df["duracao_score"] = normalizar_invertido(df["duracao_min"])
    df["escalas_score"] = normalizar_invertido(df["escalas"])
    df["hora_partida"] = df["partida"].dt.hour
    df["conveniencia_score"] = df["hora_partida"].apply(calcular_conveniencia)

    df["pontuacao"] = (
        df["preco_score"] * pesos["preco"]
        + df["duracao_score"] * pesos["duracao"]
        + df["escalas_score"] * pesos["escalas"]
        + df["conveniencia_score"] * pesos["conveniencia"]
    )

    return df.sort_values("pontuacao", ascending=False).reset_index(drop=True)

In [11]:
df_opcoes = buscar_aeroportos_alternativos("GRU", ["MXP", "LIN", "BGY"], "2026-09-15", "2026-09-30")
df_opcoes

,companhia,origem,destino,partida,preco,moeda,escalas,duracao_min,duracao,link
0,TAP Air Portugal,GRU,MXP,2026-09-24 15:30:00-03:00,801,usd,1,870,14h30,https://www.aviasales.com/search/GRU2409MXP1?t...
1,TAP Air Portugal,GRU,MXP,2026-09-25 00:45:00-03:00,801,usd,1,1755,29h15,https://www.aviasales.com/search/GRU2509MXP1?t...
2,TAP Air Portugal,GRU,MXP,2026-09-26 20:45:00-03:00,801,usd,1,1385,23h05,https://www.aviasales.com/search/GRU2609MXP1?t...
3,TAP Air Portugal,GRU,MXP,2026-09-27 20:45:00-03:00,801,usd,1,880,14h40,https://www.aviasales.com/search/GRU2709MXP1?t...
4,ITA Airways,GRU,LIN,2026-09-24 18:55:00-03:00,895,usd,1,1095,18h15,https://www.aviasales.com/search/GRU2409LIN1?t...
5,TAP Air Portugal,GRU,LIN,2026-09-27 20:45:00-03:00,943,usd,1,2415,40h15,https://www.aviasales.com/search/GRU2709LIN1?t...
6,ITA Airways,GRU,LIN,2026-09-25 18:55:00-03:00,958,usd,1,1215,20h15,https://www.aviasales.com/search/GRU2509LIN1?t...
7,Iberia,GRU,LIN,2026-09-26 14:00:00-03:00,1000,usd,1,960,16h00,https://www.aviasales.com/search/GRU2609LIN1?t...
8,TAP Air Portugal,GRU,BGY,2026-09-27 20:45:00-03:00,1068,usd,1,2120,35h20,https://www.aviasales.com/search/GRU2709BGY1?t...
9,LATAM Airlines Group,GRU,BGY,2026-09-24 15:15:00-03:00,1130,usd,2,2980,49h40,https://www.aviasales.com/search/GRU2409BGY1?t...


In [12]:
df_ranking = calcular_ranking(df_opcoes)

print("💰 Menor preço:")
display(df_opcoes.sort_values("preco").head(3)[["companhia","destino","preco","escalas","duracao"]])

print("\n🏆 Melhor custo-benefício:")
display(df_ranking.head(3)[["companhia","destino","preco","escalas","duracao","pontuacao"]])

💰 Menor preço:


,companhia,destino,preco,escalas,duracao
0,TAP Air Portugal,MXP,801,1,14h30
1,TAP Air Portugal,MXP,801,1,29h15
2,TAP Air Portugal,MXP,801,1,23h05



🏆 Melhor custo-benefício:


,companhia,destino,preco,escalas,duracao,pontuacao
0,TAP Air Portugal,MXP,801,1,14h30,1.000000
1,TAP Air Portugal,MXP,801,1,14h40,0.998815
2,ITA Airways,LIN,895,1,18h15,0.946079


In [13]:
pesos_conforto = {"preco": 0.25, "duracao": 0.35, "escalas": 0.30, "conveniencia": 0.10}

df_ranking_conforto = calcular_ranking(df_opcoes, pesos=pesos_conforto)
display(df_ranking_conforto.head(3)[["companhia","destino","preco","escalas","duracao","pontuacao"]])

,companhia,destino,preco,escalas,duracao,pontuacao
0,TAP Air Portugal,MXP,801,1,14h30,1.000000
1,TAP Air Portugal,MXP,801,1,14h40,0.998341
2,Iberia,LIN,1000,1,16h00,0.956214


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os
PASTA_PROJETO = "/content/drive/MyDrive/FlightDealFinder"
os.makedirs(PASTA_PROJETO, exist_ok=True)

DB_PATH = f"{PASTA_PROJETO}/flight_deal_finder.db"

Mounted at /content/drive


In [15]:
import sqlite3

def criar_banco():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS voos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            data_consulta TEXT,
            origem TEXT,
            destino TEXT,
            data_voo TEXT,
            companhia TEXT,
            preco REAL,
            moeda TEXT,
            duracao_min INTEGER,
            escalas INTEGER,
            aeroporto_origem TEXT,
            aeroporto_destino TEXT,
            bagagem TEXT
        )
    """)
    conn.commit()
    conn.close()

criar_banco()

In [16]:
from datetime import datetime

def salvar_no_banco(df):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    agora = datetime.now().isoformat()

    for _, voo in df.iterrows():
        cursor.execute("""
            INSERT INTO voos (
                data_consulta, origem, destino, data_voo, companhia,
                preco, moeda, duracao_min, escalas,
                aeroporto_origem, aeroporto_destino, bagagem
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            agora, voo["origem"], voo["destino"], voo["partida"].isoformat(),
            voo["companhia"], voo["preco"], voo["moeda"], voo["duracao_min"],
            voo["escalas"], voo["origem"], voo["destino"], None,
        ))

    conn.commit()
    conn.close()
    print(f"{len(df)} voos salvos no banco.")

def carregar_historico():
    conn = sqlite3.connect(DB_PATH)
    df_historico = pd.read_sql_query("SELECT * FROM voos", conn)
    conn.close()
    return df_historico

In [17]:
salvar_no_banco(df_opcoes)
carregar_historico()

12 voos salvos no banco.


,id,data_consulta,origem,destino,data_voo,companhia,preco,moeda,duracao_min,escalas,aeroporto_origem,aeroporto_destino,bagagem,link
0,1,2026-09-07T18:38:48.741014,GRU,BGY,2026-09-30T12:30:00-03:00,LA,581.0,usd,1755,2,GRU,BGY,None,None
1,2,2026-09-07T18:38:48.741014,GRU,MXP,2026-09-30T22:20:00-03:00,UA,604.0,usd,2240,2,GRU,MXP,None,None
2,3,2026-09-07T18:38:48.741014,GRU,BGY,2026-09-21T21:30:00-03:00,UA,616.0,usd,2335,3,GRU,BGY,None,None
3,4,2026-09-07T18:38:48.741014,GRU,MXP,2026-09-21T21:30:00-03:00,UA,619.0,usd,2200,2,GRU,MXP,None,None
4,5,2026-09-07T18:38:48.741014,GRU,BGY,2026-09-15T21:30:00-03:00,UA,620.0,usd,2780,3,GRU,BGY,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,265,2026-09-23T22:10:17.935196,GRU,LIN,2026-09-26T14:00:00-03:00,Iberia,1000.0,usd,960,1,GRU,LIN,None,None
265,266,2026-09-23T22:10:17.935196,GRU,BGY,2026-09-27T20:45:00-03:00,TAP Air Portugal,1068.0,usd,2120,1,GRU,BGY,None,None
266,267,2026-09-23T22:10:17.935196,GRU,BGY,2026-09-24T15:15:00-03:00,LATAM Airlines Group,1130.0,usd,2980,2,GRU,BGY,None,None
267,268,2026-09-23T22:10:17.935196,GRU,BGY,2026-09-25T14:00:00-03:00,Iberia,1192.0,usd,1520,1,GRU,BGY,None,None


In [18]:
def calcular_estatisticas(origem, destino):
    conn = sqlite3.connect(DB_PATH)
    query = "SELECT preco FROM voos WHERE aeroporto_origem = ? AND aeroporto_destino = ?"
    df = pd.read_sql_query(query, conn, params=(origem, destino))
    conn.close()

    if df.empty:
        return None

    return {
        "media": df["preco"].mean(),
        "mediana": df["preco"].median(),
        "minimo": df["preco"].min(),
        "maximo": df["preco"].max(),
        "percentil_25": df["preco"].quantile(0.25),
        "percentil_75": df["preco"].quantile(0.75),
        "desvio_padrao": df["preco"].std(),
        "quantidade": len(df),
    }

In [19]:
LIMIAR_OFERTA_EXCELENTE = -0.15   # 15% ou mais abaixo da média
LIMIAR_BOM_PRECO = -0.05          # entre 5% e 15% abaixo
LIMIAR_PRECO_ALTO = 0.10          # mais de 10% acima da média

def classificar_preco(preco_atual, media):
    diferenca = (preco_atual - media) / media

    if diferenca <= LIMIAR_OFERTA_EXCELENTE:
        classificacao = "🟢 OFERTA EXCELENTE"
    elif diferenca <= LIMIAR_BOM_PRECO:
        classificacao = "🟡 BOM PREÇO"
    elif diferenca <= LIMIAR_PRECO_ALTO:
        classificacao = "⚪ PREÇO NORMAL"
    else:
        classificacao = "🔴 PREÇO ALTO"

    return classificacao, diferenca

def relatorio_preco(origem, destino, preco_atual):
    stats = calcular_estatisticas(origem, destino)
    if stats is None:
        print("Sem histórico suficiente pra essa rota ainda.")
        return

    classificacao, diferenca = classificar_preco(preco_atual, stats["media"])

    print(f"Preço médio histórico: {stats['media']:.2f}")
    print(f"Preço atual:           {preco_atual:.2f}")
    print(f"Diferença:             {diferenca*100:+.1f}%")
    print(f"Classificação:         {classificacao}")
    print(f"(baseado em {stats['quantidade']} registros)")

In [20]:
relatorio_preco("GRU", "MXP", 604)

Preço médio histórico: 703.67
Preço atual:           604.00
Diferença:             -14.2%
Classificação:         🟡 BOM PREÇO
(baseado em 66 registros)


In [21]:
def detectar_ofertas(df_novo, limiar=LIMIAR_OFERTA_EXCELENTE):
    alertas = []

    for _, voo in df_novo.iterrows():
        stats = calcular_estatisticas(voo["origem"], voo["destino"])
        if stats is None or stats["quantidade"] < 3:
            continue  # poucos dados históricos pra confiar na média

        diferenca = (voo["preco"] - stats["media"]) / stats["media"]

        if diferenca <= limiar:
            alertas.append({
                "companhia": voo["companhia"],
                "origem": voo["origem"],
                "destino": voo["destino"],
                "preco": voo["preco"],
                "media_historica": stats["media"],
                "diferenca_pct": diferenca * 100,
            })

    return pd.DataFrame(alertas)

def imprimir_alertas(df_alertas):
    if df_alertas.empty:
        print("Nenhuma oferta detectada nesta busca.")
        return

    for _, alerta in df_alertas.iterrows():
        print(f"✈️ {alerta['companhia']} {alerta['origem']} → {alerta['destino']}")
        print(f"   Preço histórico médio: {alerta['media_historica']:.2f}")
        print(f"   Preço atual: {alerta['preco']:.2f}")
        print(f"   → {abs(alerta['diferenca_pct']):.1f}% abaixo da média")
        print(f"   → 🎯 POSSÍVEL OFERTA")
        print()

In [22]:
alertas = detectar_ofertas(df_opcoes)
imprimir_alertas(alertas)

Nenhuma oferta detectada nesta busca.


In [23]:
voo_teste = pd.DataFrame([{
    "companhia": "XX",
    "origem": "GRU",
    "destino": "MXP",
    "preco": 450,
}])

imprimir_alertas(detectar_ofertas(voo_teste))

✈️ XX GRU → MXP
   Preço histórico médio: 703.67
   Preço atual: 450.00
   → 36.0% abaixo da média
   → 🎯 POSSÍVEL OFERTA



In [24]:
conn = sqlite3.connect(DB_PATH)
try:
    conn.execute("ALTER TABLE voos ADD COLUMN link TEXT")
    conn.commit()
    print("Coluna 'link' adicionada.")
except sqlite3.OperationalError as e:
    print("Aviso:", e)  # normal se você rodar essa célula mais de uma vez
conn.close()

Aviso: duplicate column name: link


In [25]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

def _criar_animacao_radar(df, titulo="🛰️ RECON-1 — Varredura de rotas", persistencia=0.75):
    n = len(df)
    if n == 0:
        return None

    angulos = np.linspace(0, 2*np.pi, n, endpoint=False)
    preco_min, preco_max = df["preco"].min(), df["preco"].max()
    if preco_max == preco_min:
        raios = np.full(n, 0.5)
    else:
        raios = 0.15 + 0.75 * (df["preco"] - preco_min) / (preco_max - preco_min)

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"})
    fig.patch.set_facecolor("black")
    ax.set_facecolor("black")
    ax.set_ylim(0, 1)
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.grid(color="green", alpha=0.3)
    ax.set_title(titulo, color="lime", fontsize=12, pad=20)

    pontos = ax.scatter(angulos, raios, s=120, c="lime", zorder=3)
    linha_sweep, = ax.plot([], [], color="lime", linewidth=2, zorder=2)

    textos = []
    for i in range(n):
        row = df.iloc[i]
        label = f"🎯 {row['companhia']} {row['destino']}\n${row['preco']:.0f}"
        txt = ax.annotate(label, (angulos[i], raios[i]), color="lime", fontsize=7, alpha=0, ha="center")
        textos.append(txt)

    FRAMES_POR_VOLTA = 90
    DECAIMENTO = persistencia * 2 * np.pi

    def atualizar(frame):
        angulo_sweep = (frame % FRAMES_POR_VOLTA) * (2 * np.pi / FRAMES_POR_VOLTA)
        linha_sweep.set_data([angulo_sweep, angulo_sweep], [0, 1])

        distancia = (angulo_sweep - angulos) % (2 * np.pi)
        forca_sinal = np.clip(1 - distancia / DECAIMENTO, 0, 1)

        cores = np.zeros((n, 4))
        cores[:, 1] = 1.0
        cores[:, 3] = forca_sinal
        pontos.set_facecolor(cores)

        for i, txt in enumerate(textos):
            txt.set_alpha(forca_sinal[i])

        return linha_sweep, pontos

    anim = animation.FuncAnimation(fig, atualizar, frames=FRAMES_POR_VOLTA, interval=60, blit=False)
    return fig, anim

def animar_radar(df, titulo="🛰️ RECON-1 — Varredura de rotas", persistencia=0.75):
    resultado = _criar_animacao_radar(df, titulo, persistencia)
    if resultado is None:
        print("Sem alvos pra exibir no radar.")
        return None
    fig, anim = resultado
    plt.close(fig)
    return HTML(anim.to_jshtml())

def gerar_gif_radar(df, caminho="/content/radar_temp.gif", persistencia=0.75):
    resultado = _criar_animacao_radar(df, persistencia=persistencia)
    if resultado is None:
        return None
    fig, anim = resultado
    anim.save(caminho, writer="pillow", fps=15)
    plt.close(fig)
    return caminho

In [26]:
animar_radar(df_opcoes)

Output hidden; open in https://colab.research.google.com to view.

In [27]:
buscar_voos("GRU", "MXP", "2026-09")

,companhia,origem,destino,partida,preco,moeda,escalas,duracao_min,duracao,link
0,TAP Air Portugal,GRU,MXP,2026-09-24 15:30:00-03:00,4120,brl,1,870,14h30,https://www.aviasales.com/search/GRU2409MXP1?t...
1,TAP Air Portugal,GRU,MXP,2026-09-25 00:45:00-03:00,4120,brl,1,1755,29h15,https://www.aviasales.com/search/GRU2509MXP1?t...
2,TAP Air Portugal,GRU,MXP,2026-09-26 20:45:00-03:00,4120,brl,1,1385,23h05,https://www.aviasales.com/search/GRU2609MXP1?t...
3,TAP Air Portugal,GRU,MXP,2026-09-27 20:45:00-03:00,4120,brl,1,880,14h40,https://www.aviasales.com/search/GRU2709MXP1?t...


In [28]:
print(nome_companhia("G3"))
print(MAPA_COMPANHIAS.get("G3"))

Gol Transportes Aéreos
Gol Transportes Aéreos


In [29]:
!pip install "python-telegram-bot[job-queue]" nest_asyncio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 9.7 MB/s eta 0:00:00


In [30]:
TELEGRAM_TOKEN = userdata.get('TELEGRAM_TOKEN')
print("Token do bot carregado:", "OK" if TELEGRAM_TOKEN else "FALHOU")

Token do bot carregado: OK


In [31]:
!pip install "python-telegram-bot[job-queue]" nest_asyncio -q

In [32]:
resposta_companhias = requests.get("https://api.travelpayouts.com/data/en/airlines.json")
print(resposta_companhias.status_code)
print(resposta_companhias.json()[:3])

200
[{'name_translations': {'en': 'Atlas Air'}, 'code': '5Y', 'name': 'Atlas Air', 'is_lowcost': False}, {'name_translations': {'en': 'Pobeda'}, 'code': 'DP', 'name': 'Pobeda', 'is_lowcost': True}, {'name_translations': {'en': 'TAME'}, 'code': 'EQ', 'name': 'TAME', 'is_lowcost': False}]


In [33]:
conn = sqlite3.connect(DB_PATH)
df_tudo = pd.read_sql_query("SELECT * FROM voos ORDER BY data_consulta DESC", conn)
conn.close()
df_tudo

,id,data_consulta,origem,destino,data_voo,companhia,preco,moeda,duracao_min,escalas,aeroporto_origem,aeroporto_destino,bagagem,link
0,258,2026-09-23T22:10:17.935196,GRU,MXP,2026-09-24T15:30:00-03:00,TAP Air Portugal,801.0,usd,870,1,GRU,MXP,None,None
1,259,2026-09-23T22:10:17.935196,GRU,MXP,2026-09-25T00:45:00-03:00,TAP Air Portugal,801.0,usd,1755,1,GRU,MXP,None,None
2,260,2026-09-23T22:10:17.935196,GRU,MXP,2026-09-26T20:45:00-03:00,TAP Air Portugal,801.0,usd,1385,1,GRU,MXP,None,None
3,261,2026-09-23T22:10:17.935196,GRU,MXP,2026-09-27T20:45:00-03:00,TAP Air Portugal,801.0,usd,880,1,GRU,MXP,None,None
4,262,2026-09-23T22:10:17.935196,GRU,LIN,2026-09-24T18:55:00-03:00,ITA Airways,895.0,usd,1095,1,GRU,LIN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,8,2026-09-07T18:38:48.741014,GRU,MXP,2026-09-17T11:50:00-03:00,DM,700.0,usd,2965,3,GRU,MXP,None,None
265,9,2026-09-07T18:38:48.741014,GRU,LIN,2026-09-15T00:30:00-03:00,AT,710.0,usd,1620,3,GRU,LIN,None,None
266,10,2026-09-07T18:38:48.741014,GRU,LIN,2026-09-21T15:30:00-03:00,BA,715.0,usd,1025,1,GRU,LIN,None,None
267,11,2026-09-07T18:38:48.741014,GRU,LIN,2026-09-17T21:50:00-03:00,UA,720.0,usd,2570,2,GRU,LIN,None,None


In [34]:
import nest_asyncio
nest_asyncio.apply()

from datetime import datetime as dt
from telegram import Update
from telegram.ext import Application, CommandHandler, ContextTypes

def encurtar_link(url):
    try:
        resposta = requests.get("https://tinyurl.com/api-create.php", params={"url": url}, timeout=5)
        if resposta.status_code == 200:
            return resposta.text
    except requests.RequestException:
        pass
    return url

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    await update.message.reply_text(
        "🛰️ RECON-1 online. Sistemas operacionais.\n"
        "Envie /missao ORIGEM, DESTINO, DD/MM/AAAA, DD/MM/AAAA (cidade ou código) pra iniciar um reconhecimento."
    )

async def missao(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    texto = update.message.text.split(" ", 1)
    if len(texto) < 2:
        await update.message.reply_text("⚠️ Uso: /missao ORIGEM, DESTINO, DD/MM/AAAA, DD/MM/AAAA")
        return

    partes = [p.strip() for p in texto[1].split(",")]
    if len(partes) != 4:
        await update.message.reply_text(
            "⚠️ Uso: /missao ORIGEM, DESTINO, DD/MM/AAAA, DD/MM/AAAA\n"
            "Exemplo: /missao São Paulo, Milão, 15/09/2026, 30/09/2026"
        )
        return

    origem_txt, destino_txt, data_inicio_str, data_fim_str = partes
    origem = resolver_local(origem_txt)
    destino = resolver_local(destino_txt)

    if not origem or not destino:
        nao_reconhecido = origem_txt if not origem else destino_txt
        await update.message.reply_text(f"⚠️ Não reconheci '{nao_reconhecido}'. Tenta o nome da cidade ou o código IATA.")
        return

    try:
        data_inicio = dt.strptime(data_inicio_str, "%d/%m/%Y").strftime("%Y-%m-%d")
        data_fim = dt.strptime(data_fim_str, "%d/%m/%Y").strftime("%Y-%m-%d")
    except ValueError:
        await update.message.reply_text("⚠️ Data inválida. Use o formato DD/MM/AAAA.")
        return

    await update.message.reply_text(f"🛰️ Iniciando reconhecimento: {origem_txt} → {destino_txt} ({data_inicio_str} a {data_fim_str})...")

    df = buscar_datas_flexiveis(origem, destino, data_inicio, data_fim)

    if df.empty:
        await update.message.reply_text("📡 Nenhum sinal encontrado nessa varredura. Tente outra rota ou período.")
        return

    caminho_gif = gerar_gif_radar(df)
    if caminho_gif:
        with open(caminho_gif, "rb") as gif:
            await update.message.reply_animation(gif)

    df_ranking = calcular_ranking(df)
    salvar_no_banco(df)

    medalhas = ["🥇", "🥈", "🥉"]
    mensagem = "✈️ MELHORES ALVOS ENCONTRADOS\n\n"
    for i, (_, voo) in enumerate(df_ranking.head(3).iterrows()):
        medalha = medalhas[i] if i < len(medalhas) else "▫️"
        mensagem += (
            f"{medalha} {voo['companhia']} {voo['origem']} → {voo['destino']}\n"
            f"{voo['partida'].strftime('%d/%m/%Y %H:%M')}\n"
            f"{formatar_preco(voo['preco'], voo['moeda'])}\n"
            f"{voo['escalas']} escala(s) — {voo['duracao']}\n"
            f"🔗 {encurtar_link(voo['link'])}\n\n"
        )

    await update.message.reply_text(mensagem)

async def alerta(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    texto = update.message.text.split(" ", 1)
    if len(texto) < 2:
        await update.message.reply_text("⚠️ Uso: /alerta ORIGEM, DESTINO, max=VALOR")
        return

    partes = [p.strip() for p in texto[1].split(",")]
    if len(partes) != 3 or not partes[2].startswith("max="):
        await update.message.reply_text(
            "⚠️ Uso: /alerta ORIGEM, DESTINO, max=VALOR\n"
            "Exemplo: /alerta São Paulo, Milão, max=650"
        )
        return

    origem_txt, destino_txt, max_txt = partes
    origem = resolver_local(origem_txt)
    destino = resolver_local(destino_txt)

    if not origem or not destino:
        nao_reconhecido = origem_txt if not origem else destino_txt
        await update.message.reply_text(f"⚠️ Não reconheci '{nao_reconhecido}'. Tenta o nome da cidade ou o código IATA.")
        return

    try:
        preco_maximo = float(max_txt.split("=")[1])
    except ValueError:
        await update.message.reply_text("⚠️ Valor de max inválido.")
        return

    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        "INSERT INTO alertas (chat_id, origem, destino, preco_maximo, criado_em, ativo) VALUES (?, ?, ?, ?, ?, 1)",
        (update.effective_chat.id, origem, destino, preco_maximo, dt.now().isoformat())
    )
    conn.commit()
    conn.close()

    await update.message.reply_text(
        f"🎯 Alvo registrado: {origem_txt} → {destino_txt}, teto de {formatar_preco(preco_maximo)}.\n"
        f"RECON-1 vai monitorar e avisar se detectar algo dentro do orçamento."
    )

async def verificar_alertas(context: ContextTypes.DEFAULT_TYPE) -> None:
    conn = sqlite3.connect(DB_PATH)
    df_alertas = pd.read_sql_query("SELECT * FROM alertas WHERE ativo = 1", conn)
    conn.close()

    for _, a in df_alertas.iterrows():
        hoje = dt.now()
        limite = hoje + pd.Timedelta(days=60)
        df = buscar_datas_flexiveis(a["origem"], a["destino"], hoje.strftime("%Y-%m-%d"), limite.strftime("%Y-%m-%d"))

        if df.empty:
            continue

        candidatos = df[df["preco"] <= a["preco_maximo"]]
        if candidatos.empty:
            continue

        melhor = candidatos.sort_values("preco").iloc[0]
        mensagem = (
            f"🎯 ALVO DETECTADO!\n\n"
            f"{melhor['companhia']} {melhor['origem']} → {melhor['destino']}\n"
            f"{melhor['partida'].strftime('%d/%m/%Y %H:%M')}\n"
            f"{formatar_preco(melhor['preco'], melhor['moeda'])} (teto era {formatar_preco(a['preco_maximo'], melhor['moeda'])})\n"
            f"{melhor['escalas']} escala(s) — {melhor['duracao']}\n"
            f"🔗 {encurtar_link(melhor['link'])}"
        )
        await context.bot.send_message(chat_id=int(a["chat_id"]), text=mensagem)

        conn = sqlite3.connect(DB_PATH)
        conn.execute("UPDATE alertas SET ativo = 0 WHERE id = ?", (int(a["id"]),))
        conn.commit()
        conn.close()

app_telegram = Application.builder().token(TELEGRAM_TOKEN).build()
app_telegram.add_handler(CommandHandler("start", start))
app_telegram.add_handler(CommandHandler("missao", missao))
app_telegram.add_handler(CommandHandler("alerta", alerta))
app_telegram.job_queue.run_repeating(verificar_alertas, interval=1800, first=10)

print("✅ RECON-1 configurado (start, missao, alerta) com preço formatado em R$.")

✅ RECON-1 configurado (start, missao, alerta) com preço formatado em R$.


In [ ]:
print("🛰️ RECON-1 no ar.")
app_telegram.run_polling(allowed_updates=Update.ALL_TYPES)

🛰️ RECON-1 no ar.


/tmp/ipykernel_2426/603076658.py:74: UserWarning: Glyph 128752 (\N{SATELLITE}) missing from font(s) DejaVu Sans.
  anim.save(caminho, writer="pillow", fps=15)
/tmp/ipykernel_2426/603076658.py:74: UserWarning: Glyph 127919 (\N{DIRECT HIT}) missing from font(s) DejaVu Sans.
  anim.save(caminho, writer="pillow", fps=15)


2 voos salvos no banco.
